### MedAI - Data Preparation

In [22]:
from pathlib import Path
import pandas as pd
import logging
from PIL import Image

#### Logging Setup

In [23]:
LOG_FILE = Path("../logs/app.log")
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

logger.info("Data Preparation Started")

####  Dataset Paths

In [24]:
DATA_DIR = Path("../data/raw/chest_xray")

train_dir = DATA_DIR / "train"
val_dir = DATA_DIR / "val"
test_dir = DATA_DIR / "test"

print("Train Path:", train_dir)
print("Validation Path:", val_dir)
print("Test Path:", test_dir)

Train Path: ..\data\raw\chest_xray\train
Validation Path: ..\data\raw\chest_xray\val
Test Path: ..\data\raw\chest_xray\test


#### Count Images

In [25]:
def count_images(folder_path):
    count = 0

    for class_dir in folder_path.iterdir():
        if class_dir.is_dir():
            count += len(list(class_dir.glob("*")))

    return count

In [26]:

print("\nDataset Summary")
print("-" * 40)
print("Train Images:", count_images(train_dir))
print("Validation Images:", count_images(val_dir))
print("Test Images:", count_images(test_dir))

logger.info(f"Train Images: {count_images(train_dir)}")
logger.info(f"Validation Images: {count_images(val_dir)}")
logger.info(f"Test Images: {count_images(test_dir)}")


Dataset Summary
----------------------------------------
Train Images: 5216
Validation Images: 16
Test Images: 624


## Class Distribution

In [27]:
def class_distribution(folder_path):

    data = []

    for class_dir in folder_path.iterdir():

        if class_dir.is_dir():

            image_count = len(list(class_dir.glob("*")))

            data.append(
                {
                    "Class": class_dir.name,
                    "Images": image_count
                }
            )

    return pd.DataFrame(data)

In [28]:
train_df = class_distribution(train_dir)
val_df = class_distribution(val_dir)
test_df = class_distribution(test_dir)

print("Train Distribution")
display(train_df)

print("Validation Distribution")
display(val_df)

print("Test Distribution")
display(test_df)

logger.info("Class distribution generated")

Train Distribution


,Class,Images
0,NORMAL,1341
1,PNEUMONIA,3875


Validation Distribution


,Class,Images
0,NORMAL,8
1,PNEUMONIA,8


Test Distribution


,Class,Images
0,NORMAL,234
1,PNEUMONIA,390


## Check Corrupted Images

In [29]:
def check_corrupted_images(folder_path):

    corrupted = []

    for class_dir in folder_path.iterdir():

        if class_dir.is_dir():

            for img_path in class_dir.glob("*"):

                try:
                    img = Image.open(img_path)
                    img.verify()

                except Exception:
                    corrupted.append(str(img_path))

    return corrupted

In [30]:
corrupted_images = (
    check_corrupted_images(train_dir)
    + check_corrupted_images(val_dir)
    + check_corrupted_images(test_dir)
)

print(f"Corrupted Images Found: {len(corrupted_images)}")

logger.info(f"Corrupted Images Found: {len(corrupted_images)}")

Corrupted Images Found: 0


## Image Size Analysis

In [31]:
image_sizes = []

for class_dir in train_dir.iterdir():

    if class_dir.is_dir():

        for img_path in class_dir.glob("*"):

            try:

                img = Image.open(img_path)

                image_sizes.append(img.size)

            except:
                pass

sizes_df = pd.DataFrame(
    image_sizes,
    columns=["Width", "Height"]
)

display(sizes_df.describe())

,Width,Height
count,5216.000000,5216.000000
mean,1320.610813,968.074770
std,355.298743,378.855691
min,384.000000,127.000000
25%,1056.000000,688.000000
50%,1284.000000,888.000000
75%,1552.000000,1187.750000
max,2916.000000,2663.000000


In [32]:
logger.info("Data Preparation Completed Successfully")

print("✅ Data Preparation Completed")

✅ Data Preparation Completed


## Labeling

In [37]:
from pathlib import Path
import shutil
import logging

In [34]:
RAW_DATA = Path("../data/raw/chest_xray")

PROCESSED_DATA = Path("../data/processed/chest_xray")

splits = ["train", "val", "test"]

In [35]:
for split in splits:

    split_dir = PROCESSED_DATA / split

    (split_dir / "NORMAL").mkdir(parents=True, exist_ok=True)

    (split_dir / "BACTERIAL_PNEUMONIA").mkdir(
        parents=True,
        exist_ok=True
    )

    (split_dir / "VIRAL_PNEUMONIA").mkdir(
        parents=True,
        exist_ok=True
    )

print("✅ Folder structure created")

✅ Folder structure created


In [38]:
for split in splits:

    print(f"\nProcessing {split}...")

    source_split = RAW_DATA / split

    # NORMAL
    normal_src = source_split / "NORMAL"

    for img_path in normal_src.glob("*"):

        shutil.copy(
            img_path,
            PROCESSED_DATA / split / "NORMAL" / img_path.name
        )

    # PNEUMONIA
    pneumonia_src = source_split / "PNEUMONIA"

    for img_path in pneumonia_src.glob("*"):

        filename = img_path.name.lower()

        if "bacteria" in filename:

            shutil.copy(
                img_path,
                PROCESSED_DATA /
                split /
                "BACTERIAL_PNEUMONIA" /
                img_path.name
            )

        elif "virus" in filename:

            shutil.copy(
                img_path,
                PROCESSED_DATA /
                split /
                "VIRAL_PNEUMONIA" /
                img_path.name
            )

logger.info("Label Mapping Completed")
print("\n✅ Images mapped successfully")


Processing train...

Processing val...

Processing test...

✅ Images mapped successfully


In [39]:
def count_images(folder):

    return len(list(folder.glob("*")))


for split in splits:

    print(f"\n{split.upper()}")

    split_dir = PROCESSED_DATA / split

    for cls in [
        "NORMAL",
        "BACTERIAL_PNEUMONIA",
        "VIRAL_PNEUMONIA"
    ]:

        count = count_images(split_dir / cls)

        print(f"{cls}: {count}")


TRAIN
NORMAL: 1341
BACTERIAL_PNEUMONIA: 2530
VIRAL_PNEUMONIA: 1345

VAL
NORMAL: 8
BACTERIAL_PNEUMONIA: 8
VIRAL_PNEUMONIA: 0

TEST
NORMAL: 234
BACTERIAL_PNEUMONIA: 242
VIRAL_PNEUMONIA: 148


In [40]:
logger.info("3-Class Dataset Creation Successful")

print("✅ 3-Class Dataset Ready")

✅ 3-Class Dataset Ready
